In [2]:
from typing import Union
import time
import string
import numpy as np
import pandas as pd


def gen_rand_strs(
    rng: np.random._generator.Generator,
    str_cnt: int,
    str_len: tuple[int, int],
    str_chars: list[str],
) -> list[str]:
    str_lens = rng.integers(low=str_len[0], high=str_len[1], size=str_cnt, endpoint=True)
    rand_strs = [''.join(rng.choice(str_chars, size=str_len)) for str_len in str_lens]
    return rand_strs


def gen_str_vals(
    size: int,
    rng: np.random._generator.Generator,
    str_cnt: int = None,
    str_len: Union[int, tuple[int, int]] = None,
    str_chars: list[str] = None,
    col_strs: list[str] = None,
) -> np.ndarray:
    if str_cnt is None:
        str_cnt = 10
    if str_len is None:
        str_len = (5, 5)
    elif type(str_len) == int:
        str_len = (str_len, str_len)
    if col_strs is None:
        if str_chars is None:
            str_chars = [c for c in string.ascii_letters + string.digits]
        col_strs = gen_rand_strs(rng, str_cnt, str_len, str_chars)
    val = rng.choice(col_strs, size=size)
    return val


def gen_ts_vals(
    size: int,
    rng: np.random._generator.Generator,
    start_date: str = None,
    end_date: str = None,
    freq: str = None,
    random: bool = False,
) -> np.ndarray:
    if start_date is None:
        start_date = '2024-01-01'
    if end_date is None:
        end_date = '2025-01-01'
    if freq is None:
        freq = 'D'
    if random is None:
        random = False
    val = pd.date_range(start_date, end_date, freq=freq, inclusive='left')[:size]
    if random:
        val = rng.choice(val, size=size)
    return val


def gen_num_vals(
    size: int,
    rng: np.random._generator.Generator,
    low: Union[int, float] = None,
    high: Union[int, float] = None,
    dtype: str = None,
) -> np.ndarray:
    if low is None:
        low = 0
    if high is None:
        high = 2
    func = rng.integers if dtype[0] == 'i' else rng.uniform
    vals = func(low=low, high=high, size=size)
    return vals


def gen_missing_vals(
    vals: np.ndarray,
    rng: np.random._generator.Generator,
    dtype: str,
    missing_pct: float = None,
) -> np.ndarray:
    if missing_pct is None or missing_pct <= 0 or missing_pct >= 1:
        return vals
    if dtype == 's':
        missing_val = None
    elif dtype == 't':
        missing_val = np.datetime64('NaT')
    else:
        missing_val = np.nan
    if dtype == 'i':
        vals = vals.astype(np.float64)
    mask = rng.uniform(size=len(vals)) <= missing_pct
    vals[mask] = missing_val
    return vals


def sanitize_parameters(
    name_prefix: str,
    params: dict,
    par_names: list[str],
):
    if type(params) == int:
        cnt = params
        par = {}
    else:
        cnt = params['count']
        par = {
            k: v + [None] * (cnt - len(v)) if type(v) == list else [v] * cnt
            for k, v in params.items()
        }
    if name_prefix in ('i', 'f'):
        par_names += ['dtype']
        par['dtype'] = [name_prefix] * cnt
    default_val = [None] * cnt
    parameters = [
        {key: par.get(key, default_val)[i] for key in par_names}
        for i in range(cnt)
    ]
    col_names = par.get('name')
    if col_names is None or col_names.count(None) > 1 or col_names.count('') > 1:
        col_names = [f'{name_prefix}{i}' for i in range(1, cnt+1)]
    col_missing_pcts = par.get('missing_pct', default_val)
    return col_names, parameters, col_missing_pcts


def gen_rand_df(
    nrow: int,
    str_cols: dict = None,
    ts_cols: dict = None,
    int_cols: dict = None,
    float_cols: dict = None,
    rand_seed: int = 11,
) -> pd.DataFrame:
    col_types = ['s', 't', 'i', 'f']
    inputs = [str_cols, ts_cols, int_cols, float_cols]
    funcs = [gen_str_vals, gen_ts_vals, gen_num_vals, gen_num_vals]
    par_names = [
        ['str_cnt', 'str_len', 'str_chars', 'col_strs'],
        ['start_date', 'end_date', 'freq', 'random'],
        ['low', 'high'],
        ['low', 'high'],
    ]
    df = pd.DataFrame()
    rng = np.random.default_rng(seed=rand_seed)
    for i, params in enumerate(inputs):
        if params is not None:
            col_names, col_params, col_missing_pcts = sanitize_parameters(
                col_types[i], params, par_names[i]
            )
            df = pd.concat([df, pd.DataFrame({
                col: gen_missing_vals(
                    funcs[i](nrow, rng, **col_params[j]),
                    rng,
                    col_types[i],
                    col_missing_pcts[j],
                ) for j, col in enumerate(col_names)
            })], axis=1)
    return df

In [8]:
pd.set_option('display.width', 240)

In [3]:
def get_df():
    return gen_rand_df(
        nrow=10,
        str_cols={
            'count': 2,
            'name': ['id', 'category'],
            'str_len': [8, (5,20)],
            'str_count': [100, 30],
        },
        ts_cols={
            'count': 2,
            'name': ['start_date', 'end_date'],
            'start_date': ['2020-01-01', '2023-01-01'],
            'end_date': ['2023-01-01', '2025-01-01'],
            'freq': 'MS',
            'random': True,
        },
        float_cols={
            'count': 2,
            'low': 0.0,
            'high': 100.0,
            'missing_pct': 0.1,
        },
    )
df = get_df()
print(df[:2])

         id    category start_date   end_date        f1         f2
0  8v5KSoKX       jIMki 2020-01-01 2023-07-01  35.20661  76.041564
1  ihXEKLSb  bws6TOEr06 2020-05-01 2023-02-01       NaN  26.725758


In [11]:
dc = df.copy()
dg = df.copy()

In [5]:
# ChatGPT solution
def generate_half_hourly(row):
    return pd.date_range(start=row['start_date'], end=row['end_date'], freq='30min')

# Explode the date range into half-hourly timestamps
dc['ts'] = dc.apply(generate_half_hourly, axis=1)

# Explode the DataFrame on the 'ts' column
dc = dc.explode('ts')
dc[:2]


,id,category,start_date,end_date,f1,f2,ts
0,8v5KSoKX,jIMki,2020-01-01,2023-07-01,35.20661,76.041564,2020-01-01 00:00:00
0,8v5KSoKX,jIMki,2020-01-01,2023-07-01,35.20661,76.041564,2020-01-01 00:30:00


In [34]:
%%timeit -r 3 -n 7
dg = df.copy()
dg['ts'] = dg.apply(lambda row: pd.date_range(row['start_date'], row['end_date'], freq='30min'), axis=1)
# dz = dg.explode('ts')

694 ms ± 12.6 ms per loop (mean ± std. dev. of 3 runs, 7 loops each)


In [25]:
%%timeit -r 3 -n 7
dz = dg.explode('ts')

691 ms ± 7.47 ms per loop (mean ± std. dev. of 3 runs, 7 loops each)


In [35]:
%%timeit -r 3 -n 7
d1 = df.copy()
d1['ts'] = [
        pd.date_range(start, end, freq='30min')
        for start, end in zip(df['start_date'].values, df['end_date'].values)
    ]
# d1 = d1.explode('ts')

684 ms ± 12.6 ms per loop (mean ± std. dev. of 3 runs, 7 loops each)


In [21]:
%%timeit -r 3 -n 7
for start, end in zip(df['start_date'].values, df['end_date'].values): pass

10.4 µs ± 2.55 µs per loop (mean ± std. dev. of 3 runs, 7 loops each)


In [22]:
351/10.4

33.75

In [77]:
def explode_old(df, start_date_col, end_date_col, freq):
    t0 = time.time()
    df['ts'] = [
        pd.date_range(start=row[start_date_col], end=row[end_date_col], freq=freq)
        for (_, row) in df.iterrows()
    ]
    print(f'Old  create list time: {time.time()-t0:.3f}')

    t0 = time.time()
    df = df.explode('ts')
    print(f'Old explode list time: {time.time()-t0:.3f}')
    return df

def explode_new(df, start_date_col, end_date_col, freq):
    # Get exploded timestamp column
    # t0 = time.time()
    dt = pd.concat([
        pd.DataFrame({'i': i, 'ts': pd.date_range(start=s, end=e, freq=freq)})
        for i, (s, e) in enumerate(zip(df[start_date_col], df[end_date_col]))
    ]).set_index('i').rename_axis(None, axis=0)
    # print(f'New  create list time: {time.time()-t0:.3f}')

    # Re-sampling df based on new timestamp column
    # t0 = time.time()
    df = df.reindex(dt.index).assign(ts=dt.ts)
    # print(f'New      reindex time: {time.time()-t0:.3f}')
    return df

In [31]:
start_date_col = 'start_date'
end_date_col = 'end_date'
freq = '30min'

In [78]:
%%timeit -r 8 -n 10
# t0 = time.time()
d2 = explode_new(df, 'start_date', 'end_date', '30min')
# t2 = time.time() - t0
# print(f'New time {t2:.3f}')

52.3 ms ± 555 µs per loop (mean ± std. dev. of 8 runs, 10 loops each)


In [33]:
t0 = time.time()
d1 = explode_old(df, 'start_date', 'end_date', '30min')
t1 = time.time() - t0
print(f'Old time {t1:.3f}')

Old  create list time: 0.962
Old explode list time: 1.018
Old time 1.981


In [7]:
%%time
df['ts'] = [
    pd.date_range(start=row[start_date_col], end=row[end_date_col], freq=freq)
    for (_, row) in df.iterrows()
]

CPU times: user 6.26 s, sys: 131 ms, total: 6.4 s
Wall time: 6.4 s


In [8]:
%%time
d = df.explode('ts')

CPU times: user 6.38 s, sys: 224 ms, total: 6.6 s
Wall time: 6.6 s


In [9]:
df[:2]

,id,category,start_date,end_date,f1,f2,ts
0,8v5KSoKX,Q7y4yRZeT,2022-07-01,2023-09-01,43.603179,NaN,"DatetimeIndex(['2022-07-01 00:00:00', '2022-07..."
1,ihXEKLSb,4Q2WJ5DjIMkiPBDWQ3kV,2022-06-01,2023-01-01,36.413329,78.649345,"DatetimeIndex(['2022-06-01 00:00:00', '2022-06..."


In [10]:
%%time
d = df.get(['ts']).reset_index(names=['id']).explode('ts')

CPU times: user 6.41 s, sys: 160 ms, total: 6.57 s
Wall time: 6.61 s


In [40]:
# %%time
d = df.get(['ts']).rename_axis('i', axis=0).reset_index()
d[:2]

,i,ts
0,0,"DatetimeIndex(['2020-01-01 00:00:00', '2020-01..."
1,1,"DatetimeIndex(['2020-05-01 00:00:00', '2020-05..."


In [49]:
%%timeit -r 10 -n 100
dt = pd.concat([
    pd.DataFrame({'i': key, 'ts': row['ts']})
    for (key, row) in d.iterrows()
]).set_index('i').rename_axis(None, axis=0)
# dt[:2]

6.84 ms ± 98.5 µs per loop (mean ± std. dev. of 10 runs, 100 loops each)


In [50]:
%%timeit -r 10 -n 100
dt = pd.concat([
    pd.DataFrame({'i': i, 'ts': ts})
    for i, ts in zip(d['i'].values, d['ts'].values)
]).set_index('i').rename_axis(None, axis=0)
# dt[:2]

6.41 ms ± 181 µs per loop (mean ± std. dev. of 10 runs, 100 loops each)


In [13]:
%%time
d = df.drop(columns='ts').reindex(dt.index)
d['ts'] = dt.ts
d[:2]

CPU times: user 103 ms, sys: 32 ms, total: 135 ms
Wall time: 133 ms


,id,category,start_date,end_date,f1,f2,ts
0,8v5KSoKX,Q7y4yRZeT,2022-07-01,2023-09-01,43.603179,NaN,2022-07-01 00:00:00
0,8v5KSoKX,Q7y4yRZeT,2022-07-01,2023-09-01,43.603179,NaN,2022-07-01 00:30:00


In [14]:
# %%timeit
# explode `ts` column
d = df.get(['ts'])
dt = pd.concat([
    pd.DataFrame({'i': key, 'ts': row['ts']})
    for (key, row) in d.iterrows()
]).set_index('i').rename_axis(None, axis=0)
d = df.drop(columns='ts').reindex(dt.index)
d['ts'] = dt.ts
d[:2]

,id,category,start_date,end_date,f1,f2,ts
0,8v5KSoKX,Q7y4yRZeT,2022-07-01,2023-09-01,43.603179,NaN,2022-07-01 00:00:00
0,8v5KSoKX,Q7y4yRZeT,2022-07-01,2023-09-01,43.603179,NaN,2022-07-01 00:30:00


In [15]:
# %%timeit
# explode `ts` column and expand
dt = pd.concat([
    pd.DataFrame({'i': key, 'ts': pd.date_range(start=row['start_date'], end=row['end_date'], freq='30min')})
    for (key, row) in df.iterrows()
]).set_index('i').rename_axis(None, axis=0)
dx = df.drop(columns='ts').reindex(dt.index)
dx['ts'] = dt.ts
dx[:2]

,id,category,start_date,end_date,f1,f2,ts
0,8v5KSoKX,Q7y4yRZeT,2022-07-01,2023-09-01,43.603179,NaN,2022-07-01 00:00:00
0,8v5KSoKX,Q7y4yRZeT,2022-07-01,2023-09-01,43.603179,NaN,2022-07-01 00:30:00


In [52]:
def explode_df_column(df):
    d = df.get(['ts']).rename_axis('i', axis=0).reset_index()
    dt = pd.concat([
        pd.DataFrame({'i': key, 'ts': row['ts']})
        for (key, row) in d.iterrows()
    ]).set_index('i').rename_axis(None, axis=0)
    df = df.drop(columns='ts').reindex(dt.index)
    df['ts'] = dt.ts
    return df
dd = df.copy()
dd['ts'] = [
    pd.date_range(start=row[start_date_col], end=row[end_date_col], freq=freq)
    for (_, row) in dd.iterrows()
]

In [55]:
# %%timeit -r 10 -n 100
dx = dd.copy()
dy = explode_df_column(dx)

In [56]:
dz = dd.explode('ts')
dz.shape

(554986, 7)

In [58]:
%%timeit -r 10 -n 100
# Create a DataFrame with new index and exploded ts column
dt = pd.concat([
    pd.DataFrame({'i': i, 'ts': pd.date_range(start, end, freq='30min')})
    for i, (start, end) in enumerate(zip(df['start_date'], df['end_date']))
]).set_index('i').rename_axis(None, axis=0)

# Resample original df based on new index and add the exploded ts column
dn = df.reindex(dt.index).assign(ts=dt.ts)

49.8 ms ± 932 µs per loop (mean ± std. dev. of 10 runs, 100 loops each)


In [60]:
(703+691)/49.8

27.991967871485944